# GridTrackNet fine-tune / train on your labelled clips

**One-time setup.** On the PC, `python finetune/ft.py colab-zip` builds `tennis_finetune_colab.zip`: the videos,
every label file and sidecar, the workspace records (`clips.csv` and `review_status.json`, which the trainer needs,
plus `holdouts.json` and `exclude.txt` when the workspace has them), the starting weights and the trainer's code.
Upload that one file to the top of your Google Drive (`MyDrive/tennis_finetune_colab.zip`).

Then: Runtime → Change runtime type → **GPU** (T4 works; L4/A100 is several times faster), and Runtime → **Run all**.
The notebook does both kinds of run: the held-out selection run (`HOLDOUT` or `VAL_CLIPS`, scored every few epochs,
`best.npz` the epoch the held-out metric prefers) and, with `SELECTION_RUN` set to that run's `TAG`, the all-clips run
that follows its recipe on every clip for the deployable weights.

What happens: the first cell mounts Drive and unpacks the zip onto the runtime's local disk (`/content/tennis_finetune`,
fast; the Drive mount is slow for many files). Everything the run produces (the log, `results.csv`, `last.npz`,
`best.npz`, `winner.json`) goes to `MyDrive/tennis_finetune_runs/<TAG>/`, so it survives the runtime being recycled.
New labels: rebuild the zip, replace it in Drive, and the first cell unpacks it again. A code or record change alone
does not need the videos again: `ft.py colab-zip --code-only` writes the small `tennis_finetune_code.zip`; upload it beside
the big one and the first cell lays it over the unpacked copy on every run.

**Saving and resuming.** After every epoch the trainer writes `last.npz` (the weights), `checkpoint.pt` (weights,
optimizer, epoch and best-so-far bookkeeping) and a `results.csv` row to the run folder on Drive, and `best.npz` whenever
the held-out detector metric improves. If Colab recycles the runtime mid-run, reopen the notebook, keep the same `TAG`
and Run all: the settings cell sees the checkpoint and the trainer continues from the next epoch. Raise `EPOCHS` the same
way to train a finished run longer. The other settings must stay as they were: the trainer refuses to resume a run
whose `config.json` disagrees with them, and says which one differs.

Labels are consumed at 30 FPS cadence: every **2nd** frame of a 60 FPS clip, every frame of a 30 FPS clip, exactly
what `pretrack.py`/`label_tool.py` produce.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive')
ZIP = DRIVE / 'tennis_finetune_colab.zip'
DATA = Path('/content/tennis_finetune')               # the unpacked workspace: videos/, labels/, code/, models/, records
WORK = DRIVE / 'tennis_finetune_runs'                  # every run's outputs, on Drive
WORK.mkdir(parents=True, exist_ok=True)
if not ZIP.is_file():
    raise SystemExit(f'Upload tennis_finetune_colab.zip to the top of your Google Drive first (looked for {ZIP})')
# The zip is unpacked once per runtime and again whenever the file in Drive changes (its size and
# modification time are kept beside the unpacked copy).
stamp = DATA / '.unpacked_from'
fingerprint = f'{ZIP.stat().st_size} {ZIP.stat().st_mtime}'
if not (stamp.is_file() and stamp.read_text() == fingerprint):
    print(f'unpacking {ZIP.stat().st_size / 1e6:.0f} MB from Drive to the local disk ...')
    subprocess.run(['rm', '-rf', str(DATA)], check=True)
    subprocess.run(['unzip', '-q', str(ZIP), '-d', '/content'], check=True)   # the zip holds tennis_finetune/...
    stamp.write_text(fingerprint)
CODE_ZIP = DRIVE / 'tennis_finetune_code.zip'          # optional: ft.py colab-zip --code-only, laid over the big zip's code and records
if CODE_ZIP.is_file():
    print(f'laying {CODE_ZIP.name} ({CODE_ZIP.stat().st_size / 1e3:.0f} KB) over the unpacked code and records')
    subprocess.run(['unzip', '-q', '-o', str(CODE_ZIP), '-d', '/content'], check=True)
REQUIRED_RECORDS = ('clips.csv', 'review_status.json')      # without them the trainer trusts no clip
OPTIONAL_RECORDS = ('holdouts.json', 'exclude.txt')         # the named held-out sets (HOLDOUT needs it), the exclusions
missing = [name for name in REQUIRED_RECORDS if not (DATA / name).is_file()]
if missing:
    raise SystemExit(f'The zip is missing the workspace records {missing}; rebuild it on the PC')
records = [name for name in REQUIRED_RECORDS + OPTIONAL_RECORDS if (DATA / name).is_file()]
VIDEO_SUFFIXES = {'.mp4', '.mov', '.mkv', '.avi', '.m4v'}
clips = sorted(p.stem for p in (DATA / 'videos').iterdir() if p.suffix.lower() in VIDEO_SUFFIXES)
labels = sorted(p.name for p in (DATA / 'labels').glob('*_ball.csv'))
print(f'{len(clips)} videos, {len(labels)} label files, records {records}')
print('models:', sorted(p.name for p in (DATA / 'models').glob('*.npz')))
print('earlier runs on Drive:', sorted(p.name for p in WORK.iterdir() if p.is_dir()))

In [ ]:
# The trainer and everything it imports from the repo come in the zip under code/.
# REQUIRED is the trainer's internal import closure (tests/test_colab_notebook.py checks it against the code).
CODE = DATA / 'code'
REQUIRED = [
    'finetune/__init__.py', 'finetune/training/__init__.py', 'finetune/training/train_gridtracknet.py',
    'finetune/training/loop.py', 'finetune/training/metrics.py',
    'finetune/training/clips.py', 'finetune/training/frame_cache.py', 'finetune/training/dataset.py',
    'finetune/data/__init__.py', 'finetune/data/data_policy.py', 'finetune/data/fileio.py', 'finetune/data/labels.py',
    'finetune/data/detector_metrics.py', 'finetune/data/workspace.py', 'finetune/data/detect.py', 'finetune/data/holdouts.py',
    'tennis_tracker/__init__.py', 'tennis_tracker/gridtracknet.py', 'tennis_tracker/frame_rate.py',
    'tennis_tracker/config.py', 'tennis_tracker/paths.py', 'tennis_tracker/engines/__init__.py',
    'tennis_tracker/engines/ball_detectors.py',
    'tennis_tracker/engines/engine_boxes.py', 'tennis_tracker/engines/engine_session.py', 'tennis_tracker/engines/engine_files.py',
    'ball_in_play_selector/__init__.py', 'ball_in_play_selector/association.py', 'ball_in_play_selector/config.py',
    'ball_in_play_selector/core.py', 'ball_in_play_selector/models.py', 'ball_in_play_selector/physics.py',
    'ball_in_play_selector/scoring.py', 'ball_in_play_selector/utils.py',
]
missing = [name for name in REQUIRED if not (CODE / name).is_file()]
if missing:
    raise SystemExit(f'The zip is missing code files {missing}; rebuild it on the PC with the current repo')
!pip -q install opencv-python-headless numpy
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
#@title Training settings
MODE = "scratch"            #@param ["finetune", "scratch"]
# The fine-tune start; a scratch run still uses it for the hard-negative prepass.
START_WEIGHTS = "gridtracknet_weights_torch.npz"  #@param {type:"string"}
# A held-out set from holdouts.json (whole recordings); empty = use VAL_CLIPS.
HOLDOUT = "mixed_v1"        #@param {type:"string"}
# Space-separated clip stems to hold out when HOLDOUT is empty; empty = video10.
VAL_CLIPS = ""              #@param {type:"string"}
# The TAG of a finished held-out run on Drive: set, this is the all-clips run that follows that run's recipe on every clip, nothing held out or scored (HOLDOUT, VAL_CLIPS and SCORE_EVERY are not used).
SELECTION_RUN = ""          #@param {type:"string"}
# A ceiling, not a target: best.npz is kept by the held-out metric, so stop (or resume later) once the table has plateaued. With SELECTION_RUN, 0 = that run's best epoch.
EPOCHS = 80                 #@param {type:"integer"}
BATCH_SIZE = 4              #@param {type:"integer"}
# Adadelta; 0.3 for a gentle fine-tune.
LR = 1.0                    #@param {type:"number"}
# The augmentation and sampling seed; the recipe runs used 7 (11 to repeat).
SEED = 7                    #@param {type:"integer"}
# Space-separated, any of: basic strong translate reverse compress.
AUGMENT = "basic"           #@param {type:"string"}
# Score the held-out set every K epochs (a few minutes each); best.npz is the epoch the detector metric prefers.
SCORE_EVERY = 2             #@param {type:"integer"}
# Sampling weight of units holding a no-ball frame (3 was the recipe's false-alarm lever R4a; unmeasured).
OVERSAMPLE_ABSENT = 1.0     #@param {type:"number"}
# Data-loading processes; 0 = one per CPU the runtime has (the GPU otherwise waits on PNG decoding).
WORKERS = 0                 #@param {type:"integer"}
# The run's folder under MyDrive/tennis_finetune_runs/; one that holds a checkpoint.pt is resumed.
TAG = "scratch1"            #@param {type:"string"}

import os
RUN = WORK / TAG
all_clips = bool(SELECTION_RUN.strip())
if all_clips:
    SELECTION = WORK / SELECTION_RUN.strip()
    if SELECTION_RUN.strip() == TAG:
        raise SystemExit(f'SELECTION_RUN and TAG are both {TAG!r}: the all-clips run needs a folder of its own')
    if not (SELECTION / 'winner.json').is_file():
        raise SystemExit(f'SELECTION_RUN {SELECTION_RUN!r} is not a finished run on Drive: no {SELECTION / "winner.json"}')
RUN.mkdir(parents=True, exist_ok=True)
workers = WORKERS or max(1, os.cpu_count() or 1)
print(f'{os.cpu_count()} CPUs, {workers} loader workers, GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
# Frames are cached on the fast local disk, not in Drive. The trainer runs as a module from code/,
# so its imports of finetune.* and tennis_tracker.* resolve to the staged copies; -u keeps its
# log unbuffered through the pipe the next cell reads it from.
cmd = [sys.executable, '-u', '-m', 'finetune.training.train_gridtracknet',
       '--videos', str(DATA / 'videos'), '--labels', str(DATA / 'labels'),
       '--weights', str(DATA / 'models' / START_WEIGHTS),
       '--save', str(RUN / 'winner.npz'),
       '--data-dir', '/content/cache/data', '--run-dir', str(RUN),
       '--batch-size', str(BATCH_SIZE), '--lr', str(LR),
       '--workers', str(workers), '--prep-workers', str(workers),
       '--seed', str(SEED), '--augment', *AUGMENT.split(),
       '--oversample-absent', str(OVERSAMPLE_ABSENT)]
if not (all_clips and EPOCHS == 0):          # 0 with SELECTION_RUN: the trainer takes the selection run's best epoch
    cmd += ['--epochs', str(EPOCHS)]
if all_clips:
    cmd += ['--all-clips', '--selection-run', str(SELECTION)]
    print(f'all-clips run: every clip, nothing held out, following the recipe of {SELECTION_RUN.strip()}'
          + (f' for {EPOCHS} epochs' if EPOCHS else ' for its best epoch count'))
else:
    cmd += ['--score-every', str(SCORE_EVERY)]
    if HOLDOUT.strip():
        cmd += ['--holdout', HOLDOUT.strip()]
    elif VAL_CLIPS.strip():
        cmd += ['--val-clips', *VAL_CLIPS.split()]
    held_out = f'HOLDOUT {HOLDOUT.strip()}' if HOLDOUT.strip() else f'VAL_CLIPS {VAL_CLIPS.split() or ["video10"]}'
    print(f'held-out selection run: {held_out}, scored every {SCORE_EVERY} epochs')
if MODE == 'scratch':
    cmd.append('--from-scratch')
if (RUN / 'checkpoint.pt').is_file():
    cmd.append('--resume')
    print(f'resuming {TAG} from its checkpoint on Drive (delete {RUN} or change TAG for a fresh run)')
print(' '.join(cmd))

In [ ]:
# Runs the trainer and streams its log here and to Drive (RUN/colab.log). The trainer prints a step line every few
# seconds (loss, it/s, eta) and, after every epoch, one summary line: train loss, held-out recall / wrong / false alarm
# and which epoch best.npz holds. On a scored epoch the held-out table so far is reprinted underneath, with the
# epoch best.npz holds marked. An all-clips run (SELECTION_RUN set) holds nothing out, so its epoch lines carry the
# train loss alone and no table follows. Interrupting the cell stops the trainer too; the run resumes from its checkpoint.
import csv, re, subprocess

def held_out_table(best_epoch):
    """The scored rows of results.csv so far (an unscored epoch leaves the metric columns empty), or
    what went wrong reading them: a table that cannot be read must never stop the log."""
    try:
        with (RUN / 'results.csv').open(newline='') as handle:
            scored = [row for row in csv.DictReader(handle) if row.get('recall')]
        lines = ['    epoch  train_loss  recall  wrong  falseAlarm  med_px  min']
        for row in scored:
            mark = ' <- best.npz' if int(row['epoch']) == best_epoch else ''
            lines.append(f"    {int(row['epoch']):5d}  {float(row['train_loss']):.6f}  {float(row['recall']):.3f}"
                         f"  {float(row['wrong']):.3f}  {float(row['false_alarm']):.3f}      {float(row['median_px']):.1f}"
                         f"  {float(row['seconds']) / 60:3.0f}{mark}")
        return '\n'.join(lines)
    except (OSError, ValueError, KeyError) as error:
        return f'    (results.csv not readable yet: {error})'

process = subprocess.Popen(cmd, cwd=str(CODE), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    with (RUN / 'colab.log').open('a', encoding='utf-8') as log:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
            best = re.search(r'best epoch (\d+)', line)
            if line.startswith('[train] epoch') and best:
                print(held_out_table(int(best.group(1))), flush=True)
finally:
    if process.poll() is None:
        process.terminate()
    process.wait()
print('exit code', process.returncode)

In [ ]:
# The whole run at a glance: the per-epoch table, the curves and the winner decision. An all-clips run has only the
# train loss to show: nothing was held out, so results.csv has no validation loss and no scored rows.
import json
import pandas as pd
import matplotlib.pyplot as plt
results = pd.read_csv(RUN / 'results.csv')
print(results.to_string(index=False))
scored = results.dropna(subset=['recall'])
fig, axes = plt.subplots(1, 3 if len(scored) else 1, figsize=(15 if len(scored) else 5, 3.5), squeeze=False)
axes = axes[0]
axes[0].plot(results['epoch'], results['train_loss'], label='train')
if results['val_loss'].notna().any():
    axes[0].plot(results['epoch'], results['val_loss'], label='val')
axes[0].set_title('loss'); axes[0].legend()
if len(scored):
    axes[1].plot(scored['epoch'], scored['recall'], marker='o'); axes[1].set_title('held-out recall')
    axes[2].plot(scored['epoch'], scored['false_alarm'], marker='o', label='false alarm')
    axes[2].plot(scored['epoch'], scored['wrong'], marker='o', label='wrong')
    axes[2].set_title('held-out false alarm / wrong'); axes[2].legend()
for ax in axes:
    ax.set_xlabel('epoch'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
if (RUN / 'winner.json').is_file():
    print(json.dumps(json.loads((RUN / 'winner.json').read_text()), indent=2))
else:
    print('no winner.json yet: the run has not reached its last epoch (resume it with the same TAG)')
if SELECTION_RUN.strip():
    print(f'\nWeights on Drive: {RUN}/winner.npz (last.npz, the final epoch, trained on every clip following {SELECTION_RUN.strip()}).')
    print('They saw every labelled clip, so no held-out number can be put on them: the test left is the tracker on unlabelled footage.')
else:
    print(f'\nWeights on Drive: {RUN}/best.npz (the epoch the detector metric chose), last.npz (the final epoch), winner.npz (the saved result).')
    print('Back on the PC, score any of them with  evaluate_archive.py --archive finetune --mode raw --holdout mixed_v1 --gridtracknet-weights <file>')

## After the run

1. Download `best.npz` from `MyDrive/tennis_finetune_runs/<TAG>/` and score it on the PC against the bundled weights on
   the same held-out set: `evaluate_archive.py --archive finetune --mode raw --holdout mixed_v1 --gridtracknet-weights <best.npz>`,
   and the same command without `--gridtracknet-weights` for the bundled weights; `evaluation/compare_eval_logs.py` sets
   the two logs side by side. The acceptance rule is in `audit/gridtracknet_recipe_experiments.md`.
2. The deployable weights come from the trainer's `--all-clips` mode: nothing held out, the chosen recipe on every
   clip. Run it in this notebook: set `SELECTION_RUN` to the held-out run's `TAG`, give the run a new `TAG` (say
   `scratch2_all`), set `EPOCHS` to 0 (the held-out run's best epoch) and Run all. The result is `winner.npz` in
   `MyDrive/tennis_finetune_runs/<new TAG>/`. Those weights saw every labelled clip, so they cannot be measured on the
   held-out clips any more; the only test left is running the tracker on unlabelled footage.
3. `ft.py promote` copies a winner over `models/gridtracknet_weights_torch.npz` with a backup; that is your call.